# Flujo Completo End-to-End: De los Datos a las Decisiones

**Módulo 04** | **Sesión 8** | **Duración estimada: 2h**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-04-predictivo-proyecto/notebooks/04_04_flujo_completo.ipynb)

## Objetivos de Aprendizaje

| # | Competencia | Descripción | Nivel |
|---|-------------|-------------|-------|
| 1 | Flujo end-to-end | Ejecutar el ciclo completo: Importar → Limpiar → Explorar → Modelar → Evaluar → Comunicar | Aplicación |
| 2 | Integración de habilidades | Combinar técnicas de los módulos anteriores en un solo análisis coherente | Síntesis |
| 3 | Comunicación de resultados | Traducir coeficientes y métricas a hallazgos comprensibles para audiencias no técnicas | Evaluación |
| 4 | Pensamiento crítico | Identificar limitaciones del modelo y proponer mejoras | Análisis |

## ¿Por qué un flujo completo?

En las sesiones anteriores trabajamos cada etapa por separado: limpieza, exploración, modelado, evaluación. En la práctica real, estas etapas **no ocurren de forma aislada** — forman un ciclo continuo donde cada paso informa al siguiente.

En esta sesión vamos a recorrer el flujo completo con un solo dataset, desde la carga de datos hasta la comunicación de hallazgos. El objetivo es que al terminar tengas un **notebook de referencia** que puedas adaptar para tu proyecto integrador.

### Las 6 etapas del flujo

```
1. Importar → 2. Limpiar → 3. Explorar → 4. Modelar → 5. Evaluar → 6. Comunicar
```

---

## Paso 1: Importar datos

Cargamos el dataset de rendimiento académico y hacemos una inspección inicial.

In [ ]:
# Librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Configuración visual
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Librerías cargadas correctamente')

In [ ]:
# Cargar el dataset
df = pd.read_csv('../datasets/universidad/rendimiento_academico.csv')

print(f'Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'\nColumnas disponibles:')
print(list(df.columns))
df.head()

In [ ]:
# Inspección inicial: tipos de datos y valores nulos
print('Tipos de datos:')
print(df.dtypes)
print(f'\nValores nulos por columna:')
print(df.isnull().sum())
print(f'\nEstadísticas descriptivas:')
df.describe().round(2)

---

## Paso 2: Limpieza

Verificamos y tratamos:
- Valores faltantes
- Tipos de datos incorrectos
- Outliers evidentes

In [ ]:
# Verificar valores faltantes
nulos = df.isnull().sum()
nulos_pct = (df.isnull().sum() / len(df) * 100).round(2)
resumen_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje': nulos_pct})
print('Resumen de valores faltantes:')
resumen_nulos[resumen_nulos['Nulos'] > 0]

In [ ]:
# Tratar faltantes: rellenar numéricos con la mediana, categóricos con la moda
df_limpio = df.copy()

for col in df_limpio.select_dtypes(include=['float64', 'int64']).columns:
    if df_limpio[col].isnull().sum() > 0:
        mediana = df_limpio[col].median()
        df_limpio[col].fillna(mediana, inplace=True)
        print(f'  {col}: {df[col].isnull().sum()} nulos rellenados con mediana = {mediana}')

for col in df_limpio.select_dtypes(include=['object', 'bool']).columns:
    if df_limpio[col].isnull().sum() > 0:
        moda = df_limpio[col].mode()[0]
        df_limpio[col].fillna(moda, inplace=True)
        print(f'  {col}: {df[col].isnull().sum()} nulos rellenados con moda = {moda}')

print(f'\nNulos restantes: {df_limpio.isnull().sum().sum()}')

In [ ]:
# Detección de outliers con boxplots para variables numéricas clave
vars_numericas = ['promedio', 'asistencia_pct', 'edad', 'materias_inscritas']
vars_disponibles = [v for v in vars_numericas if v in df_limpio.columns]

fig, axes = plt.subplots(1, len(vars_disponibles), figsize=(4 * len(vars_disponibles), 5))
if len(vars_disponibles) == 1:
    axes = [axes]

for ax, col in zip(axes, vars_disponibles):
    ax.boxplot(df_limpio[col].dropna(), vert=True)
    ax.set_title(col)

plt.suptitle('Detección de outliers', y=1.02)
plt.tight_layout()
plt.show()

---

## Paso 3: Exploración rápida (EDA)

Antes de modelar, necesitamos entender la estructura de los datos: distribuciones y correlaciones.

In [ ]:
# Distribución de la variable objetivo: promedio
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_limpio['promedio'], bins=25, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(df_limpio['promedio'].mean(), color='red', linestyle='--', label=f'Media: {df_limpio["promedio"].mean():.2f}')
axes[0].set_xlabel('Promedio')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución del promedio académico')
axes[0].legend()

# Promedio por condición de empleo y beca
if 'trabaja' in df_limpio.columns and 'beca' in df_limpio.columns:
    grupos = df_limpio.groupby(['trabaja', 'beca'])['promedio'].mean().unstack()
    grupos.plot(kind='bar', ax=axes[1], color=['#e74c3c', '#2ecc71'])
    axes[1].set_title('Promedio según empleo y beca')
    axes[1].set_xlabel('Trabaja')
    axes[1].set_ylabel('Promedio medio')
    axes[1].legend(title='Beca')
    axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlaciones
cols_numericas = df_limpio.select_dtypes(include=['float64', 'int64']).columns
correlaciones = df_limpio[cols_numericas].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlaciones, annot=True, cmap='RdBu_r', center=0, fmt='.2f',
            square=True, linewidths=0.5)
plt.title('Matriz de correlaciones')
plt.tight_layout()
plt.show()

# Correlaciones con el promedio
print('Correlación con promedio:')
print(correlaciones['promedio'].sort_values(ascending=False).to_string())

---

## Paso 4: Modelar

**Pregunta:** ¿Podemos predecir el promedio de un estudiante a partir de sus horas de estudio, condición de empleo y beca?

Usaremos regresión lineal múltiple — el modelo más simple e interpretable.

In [ ]:
# Preparar variables
# Convertir booleanos a numéricos si es necesario
df_modelo = df_limpio.copy()
if df_modelo['trabaja'].dtype == bool or df_modelo['trabaja'].dtype == 'object':
    df_modelo['trabaja_num'] = df_modelo['trabaja'].astype(int)
else:
    df_modelo['trabaja_num'] = df_modelo['trabaja']

if df_modelo['beca'].dtype == bool or df_modelo['beca'].dtype == 'object':
    df_modelo['beca_num'] = df_modelo['beca'].astype(int)
else:
    df_modelo['beca_num'] = df_modelo['beca']

# Variables predictoras
features = ['asistencia_pct', 'trabaja_num', 'beca_num']
# Agregar horas_estudio si existe en el dataset
if 'horas_estudio' in df_modelo.columns:
    features = ['horas_estudio'] + features

X = df_modelo[features]
y = df_modelo['promedio']

print(f'Variable objetivo: promedio')
print(f'Variables predictoras: {features}')
print(f'Observaciones: {len(X)}')

In [ ]:
# Dividir en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Entrenamiento: {len(X_train)} observaciones ({len(X_train)/len(X)*100:.0f}%)')
print(f'Prueba:        {len(X_test)} observaciones ({len(X_test)/len(X)*100:.0f}%)')

In [ ]:
# Entrenar el modelo
modelo = LinearRegression()
modelo.fit(X_train, y_train)

# Coeficientes
coeficientes = pd.DataFrame({
    'Variable': features,
    'Coeficiente': modelo.coef_
})
coeficientes = coeficientes.sort_values('Coeficiente', ascending=False)

print(f'Intercepto: {modelo.intercept_:.4f}')
print(f'\nCoeficientes del modelo:')
print(coeficientes.to_string(index=False))

---

## Paso 5: Evaluar

Calculamos las métricas clave y analizamos los residuos.

In [ ]:
# Predicciones
y_pred_train = modelo.predict(X_train)
y_pred_test = modelo.predict(X_test)

# Métricas
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)
rmse_test = np.sqrt(mse_test)

print('Métricas del modelo')
print('=' * 45)
print(f'R² entrenamiento: {r2_train:.4f}')
print(f'R² prueba:        {r2_test:.4f}')
print(f'MSE (prueba):     {mse_test:.4f}')
print(f'RMSE (prueba):    {rmse_test:.4f} puntos')
print()
if abs(r2_train - r2_test) < 0.05:
    print('Los R² de train y test son similares — el modelo generaliza bien.')
else:
    print('Hay diferencia notable entre train y test — posible sobreajuste.')

In [ ]:
# Visualización: Predicho vs Real y Residuos
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Predicho vs Real
axes[0].scatter(y_test, y_pred_test, alpha=0.5, color='steelblue', edgecolors='white')
lim_min = min(y_test.min(), y_pred_test.min())
lim_max = max(y_test.max(), y_pred_test.max())
axes[0].plot([lim_min, lim_max], [lim_min, lim_max], 'r--', linewidth=2, label='Predicción perfecta')
axes[0].set_xlabel('Promedio real')
axes[0].set_ylabel('Promedio predicho')
axes[0].set_title('Predicho vs Real')
axes[0].legend()

# Residuos
residuos = y_test - y_pred_test
axes[1].scatter(y_pred_test, residuos, alpha=0.5, color='coral', edgecolors='white')
axes[1].axhline(y=0, color='black', linewidth=1, linestyle='--')
axes[1].set_xlabel('Promedio predicho')
axes[1].set_ylabel('Residuo (Real - Predicho)')
axes[1].set_title('Distribución de residuos')

plt.tight_layout()
plt.show()

print('Izquierda: los puntos cerca de la diagonal roja indican buenas predicciones.')
print('Derecha: residuos dispersos aleatoriamente alrededor de 0 indican un modelo adecuado.')

---

## Paso 6: Comunicar

La parte más importante (y la que más se omite): **traducir los números a hallazgos comprensibles**.

Un modelo no tiene valor si sus resultados no pueden ser entendidos y utilizados por quienes toman decisiones.

In [ ]:
# Interpretar coeficientes en lenguaje no técnico
print('INTERPRETACIÓN DE RESULTADOS')
print('=' * 60)
print()
print(f'El modelo explica el {r2_test*100:.1f}% de la variación en el promedio')
print(f'académico de los estudiantes.')
print()
print('Factores analizados y su efecto:')
print('-' * 60)

for _, row in coeficientes.iterrows():
    var = row['Variable']
    coef = row['Coeficiente']
    if var == 'horas_estudio':
        print(f'  - Horas de estudio: cada hora adicional de estudio semanal')
        print(f'    se asocia con un cambio de {coef:.3f} puntos en el promedio.')
    elif var == 'asistencia_pct':
        print(f'  - Asistencia: cada punto porcentual adicional de asistencia')
        print(f'    se asocia con un cambio de {coef:.4f} puntos en el promedio.')
    elif var == 'trabaja_num':
        signo = 'mayor' if coef > 0 else 'menor'
        print(f'  - Empleo: los estudiantes que trabajan tienen, en promedio,')
        print(f'    un promedio {abs(coef):.3f} puntos {signo} que quienes no trabajan.')
    elif var == 'beca_num':
        signo = 'mayor' if coef > 0 else 'menor'
        print(f'  - Beca: los estudiantes becados tienen, en promedio,')
        print(f'    un promedio {abs(coef):.3f} puntos {signo} que quienes no tienen beca.')

In [ ]:
# Redactar 3 hallazgos clave
print('TRES HALLAZGOS CLAVE')
print('=' * 60)
print()

# Hallazgo 1: el factor más influyente
top_var = coeficientes.iloc[0]
print(f'1. El factor con mayor efecto positivo sobre el promedio es')
print(f'   "{top_var["Variable"]}" (coeficiente: {top_var["Coeficiente"]:.4f}).')
print(f'   Esto sugiere que políticas orientadas a mejorar este factor')
print(f'   tendrían el mayor impacto en el rendimiento estudiantil.')
print()

# Hallazgo 2: capacidad explicativa del modelo
print(f'2. El modelo explica el {r2_test*100:.1f}% de la variación en el promedio.')
if r2_test < 0.5:
    print(f'   Esto indica que hay otros factores importantes no incluidos')
    print(f'   en el análisis (motivación, calidad docente, contexto familiar).')
else:
    print(f'   Esto indica que las variables seleccionadas capturan una porción')
    print(f'   significativa de lo que determina el rendimiento académico.')
print()

# Hallazgo 3: error del modelo
print(f'3. El modelo se equivoca en promedio por {rmse_test:.2f} puntos.')
print(f'   Esto significa que si predecimos que un estudiante tendrá 15 de promedio,')
print(f'   su promedio real probablemente estará entre {15-rmse_test:.1f} y {15+rmse_test:.1f}.')

### Visualización final para la presentación

Un gráfico resumen que condense los hallazgos principales.

In [ ]:
# Gráfico resumen: importancia de variables
fig, ax = plt.subplots(figsize=(10, 5))

colores = ['#2ecc71' if c > 0 else '#e74c3c' for c in coeficientes['Coeficiente']]
bars = ax.barh(coeficientes['Variable'], coeficientes['Coeficiente'], color=colores)
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_xlabel('Efecto sobre el promedio (coeficiente)')
ax.set_title(f'Factores que influyen en el promedio académico (R² = {r2_test:.3f})')

# Anotaciones
for bar, coef in zip(bars, coeficientes['Coeficiente']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{coef:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print('Verde = efecto positivo | Rojo = efecto negativo')

---

## Ejercicio: Repite el flujo con otra pregunta

Ahora es tu turno. Usando el mismo dataset (`rendimiento_academico.csv`), elige una **pregunta diferente** y repite el flujo completo.

### Opciones sugeridas:

1. **¿Puede la edad y el semestre predecir la asistencia?** (variable objetivo: `asistencia_pct`)
2. **¿Influyen las materias inscritas y la condición laboral en el promedio?** (diferente conjunto de predictores)
3. **Plantea tu propia pregunta** relevante para tu cátedra.

### Pasos a seguir:

1. Formula tu pregunta de investigación
2. Selecciona variables predictoras y variable objetivo
3. Limpia y explora
4. Construye el modelo
5. Evalúa con R² y RMSE
6. Escribe 3 hallazgos en lenguaje no técnico

In [ ]:
# Tu pregunta de investigación:
# _______________________________________________

# Paso 1: Importar (ya tenemos df_limpio)


In [ ]:
# Paso 2-3: Limpiar y explorar


In [ ]:
# Paso 4: Modelar


In [ ]:
# Paso 5: Evaluar


In [ ]:
# Paso 6: Comunicar — escribe tus 3 hallazgos clave
# Hallazgo 1:
# Hallazgo 2:
# Hallazgo 3:


---

## Resumen

| Etapa | Qué hicimos | Herramientas |
|-------|-------------|-------------|
| **1. Importar** | Cargar CSV, inspeccionar dimensiones y tipos | `pd.read_csv`, `.shape`, `.dtypes` |
| **2. Limpiar** | Tratar nulos, verificar tipos, detectar outliers | `.fillna`, `.astype`, boxplots |
| **3. Explorar** | Distribuciones, correlaciones, segmentaciones | histogramas, `.corr()`, heatmap |
| **4. Modelar** | Regresión lineal múltiple con train/test | `LinearRegression`, `train_test_split` |
| **5. Evaluar** | R², MSE, RMSE, análisis de residuos | `r2_score`, `mean_squared_error` |
| **6. Comunicar** | Interpretar coeficientes, redactar hallazgos | Lenguaje no técnico, gráfico resumen |

Este flujo es la base de tu **proyecto integrador**. Adáptalo con el dataset y la pregunta de tu elección.

---

[Volver al Módulo 04](../README.md) | [Volver al programa principal](../../README.md)